In [1]:
from pyspark.sql import SparkSession


spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/15 20:42:16 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/15 20:42:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/15 20:42:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import pandas as pd

orders_df = pd.read_csv("orders.csv", delimiter=",", encoding="utf-8")

In [6]:
df_orders = spark.createDataFrame(orders_df)
df_orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)



/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enable

In [9]:
from pyspark.sql.functions import count

df_agg = (
    df_orders
        .groupBy("order_status")
        .agg(count("*").alias("total"))
        .orderBy("total", ascending=False)
)

df_agg.show()

+------------+-----+
|order_status|total|
+------------+-----+
|   completed|   20|
|   cancelled|    5|
|     pending|    5|
+------------+-----+



In [11]:
from pyspark.sql.functions import min

(
    df_orders
        .groupBy("order_date")
        .min()
        .orderBy("order_date")
        .withColumnRenamed("min(order_id)", "min_order_id")
        .show()
)

+-------------------+------------+----------------------+-----------+
|         order_date|min_order_id|min(order_customer_id)|min(amount)|
+-------------------+------------+----------------------+-----------+
|2024-03-01 09:15:32|        1001|                   101|     1250.0|
|2024-03-01 10:45:11|        1002|                   102|      890.5|
|2024-03-01 14:20:05|        1003|                   103|      320.0|
|2024-03-02 08:30:44|        1004|                   101|      450.0|
|2024-03-02 11:10:22|        1005|                   104|    2100.75|
|2024-03-02 13:55:17|        1006|                   105|      990.0|
|2024-03-03 09:05:33|        1007|                   102|     670.25|
|2024-03-03 12:40:10|        1008|                   106|     3100.0|
|2024-03-04 10:15:55|        1009|                   107|      540.0|
|2024-03-04 15:20:30|        1010|                   108|     1200.0|
|2024-03-05 08:50:12|        1011|                   109|      780.5|
|2024-03-05 11:30:40

In [13]:
import pandas as pd

order_items_df = pd.read_csv("order_items.csv", delimiter=",", encoding="utf-8")

df_order_items = spark.createDataFrame(order_items_df)
df_order_items.printSchema()

root
 |-- id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- product_price: double (nullable = true)



/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enable

In [14]:
df_order_items.show()

+---+--------+----------+--------+--------+-------------+
| id|order_id|product_id|quantity|subtotal|product_price|
+---+--------+----------+--------+--------+-------------+
|  1|    1001|       501|       2|  2500.0|       1250.0|
|  2|    1001|       502|       1|   890.5|        890.5|
|  3|    1002|       503|       3|   960.0|        320.0|
|  4|    1004|       504|       1|   450.0|        450.0|
|  5|    1005|       505|       2|  4201.5|      2100.75|
|  6|    1006|       506|       1|   990.0|        990.0|
|  7|    1007|       507|       2|  1340.5|       670.25|
|  8|    1008|       508|       1|  3100.0|       3100.0|
|  9|    1009|       509|       3|  1620.0|        540.0|
| 10|    1010|       510|       2|  2400.0|       1200.0|
| 11|    1011|       511|       1|   780.5|        780.5|
| 12|    1012|       512|       1|  1950.0|       1950.0|
| 13|    1013|       513|       2|  4600.0|       2300.0|
| 14|    1014|       514|       1|   410.0|        410.0|
| 15|    1015|

In [15]:
(
    df_order_items
        .select("order_id", "quantity", "subtotal")
        .groupBy("order_id")
        .sum("quantity", "subtotal")
        .orderBy("sum(subtotal)", ascending=False)
        .toDF("order_id", "order_quantity", "order_revenue")
        .show()
)

+--------+--------------+-------------+
|order_id|order_quantity|order_revenue|
+--------+--------------+-------------+
|    1027|             3|       8340.0|
|    1005|             5|       7171.5|
|    1028|             5|       7100.0|
|    1015|             4|       6681.0|
|    1008|             2|       6200.0|
|    1030|             3|      5970.75|
|    1024|             3|      5552.25|
|    1013|             2|       4600.0|
|    1023|             2|       4200.0|
|    1010|             4|       3900.0|
|    1001|             3|       3390.5|
|    1020|             5|       2800.0|
|    1021|             2|       2641.0|
|    1019|             1|       2450.0|
|    1018|             2|       2240.0|
|    1003|             4|       2040.0|
|    1012|             1|       1950.0|
|    1016|             2|       1760.0|
|    1009|             3|       1620.0|
|    1007|             2|       1340.5|
+--------+--------------+-------------+
only showing top 20 rows


In [16]:
from pyspark.sql.functions import round

(
    df_order_items
        .groupBy("order_id")
        .sum("quantity", "subtotal")
        .toDF("order_id", "order_quantity", "order_revenue")
        .withColumn("order_revenue", round("order_revenue", 2))
        .show()
)

+--------+--------------+-------------+
|order_id|order_quantity|order_revenue|
+--------+--------------+-------------+
|    1010|             4|       3900.0|
|    1002|             3|        960.0|
|    1009|             3|       1620.0|
|    1007|             2|       1340.5|
|    1005|             5|       7171.5|
|    1001|             3|       3390.5|
|    1008|             2|       6200.0|
|    1004|             1|        450.0|
|    1006|             1|        990.0|
|    1016|             2|       1760.0|
|    1012|             1|       1950.0|
|    1013|             2|       4600.0|
|    1011|             1|        780.5|
|    1018|             2|       2240.0|
|    1020|             5|       2800.0|
|    1015|             4|       6681.0|
|    1014|             1|        410.0|
|    1019|             1|       2450.0|
|    1017|             1|        950.0|
|    1025|             1|        930.0|
+--------+--------------+-------------+
only showing top 20 rows
